In [0]:
# fetch data from raw container (Auto loader)
# Load it into a dataframe
# Load back to bronze layer in delta tables (parquet file)

In [0]:
dbutils.widgets.text("SAS_TOKEN",dbutils.secrets.get(scope='Connectors', key='SAS-token'))
# mode = dbutils.widgets.get("mode") uncomment in deployment
mode = "incremental_load"

In [0]:
spark.conf.set("fs.azure.account.auth.type.bitcoindatalake.dfs.core.windows.net", "SAS")
spark.conf.set("fs.azure.sas.token.provider.type.bitcoindatalake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set("fs.azure.sas.fixed.token.bitcoindatalake.dfs.core.windows.net", dbutils.widgets.get("SAS_TOKEN"))

In [0]:
def get_chart_data(filename):
    return spark.readStream \
                    .format("cloudFiles")\
                    .option("cloudFiles.format", "json")\
                    .option("pathGlobFilter", f"*{filename}*.json")\
                    .option("cloudFiles.inferColumnTypes", "false")\
                    .option("cloudFiles.schemaLocation", f"abfss://raw@bitcoindatalake.dfs.core.windows.net/schemas/schema_{filename}/")\
                    .load("abfss://raw@bitcoindatalake.dfs.core.windows.net/")

In [0]:
from pyspark.sql.types import ArrayType, DoubleType, StructField, StructType, IntegerType, StringType, DateType, BooleanType
json_2d_schema = ArrayType(ArrayType(DoubleType()))

final_schema = StructType([
    StructField("ohlc", json_2d_schema, True)
])

In [0]:
def get_ohlc_data(filename):
    return spark.readStream \
                    .format("cloudFiles")\
                    .option("cloudFiles.format", "json")\
                    .option("pathGlobFilter", f"*{filename}*.json")\
                    .option("multiLine", "true")\
                    .schema(final_schema)\
                    .option("cloudFiles.inferColumnTypes", "false")\
                    .option("cloudFiles.schemaLocation", f"abfss://raw@bitcoindatalake.dfs.core.windows.net/schemas/schema_{filename}/")\
                    .load("abfss://raw@bitcoindatalake.dfs.core.windows.net/")

In [0]:
coin_schema = StructType([
    StructField("coin_id", StringType(), True),
    StructField("coin_key", StringType(), True),
    StructField("founded_year", DateType(), True),
    StructField("is_active", BooleanType(), True),
    StructField("source", StringType(), True),
    StructField("ticker_symbol", StringType(), True)
])

In [0]:
def get_coin(filename):
    return spark.readStream \
                    .format("cloudFiles")\
                    .option("cloudFiles.format", "json")\
                    .option("pathGlobFilter", f"*{filename}*.json")\
                    .option("cloudFiles.inferColumnTypes", "false")\
                    .schema(coin_schema)\
                    .option("cloudFiles.schemaLocation", f"abfss://raw@bitcoindatalake.dfs.core.windows.net/schemas/schema_{filename}/")\
                    .load("abfss://raw@bitcoindatalake.dfs.core.windows.net/")

In [0]:
def get_currency(filename):
    return spark.readStream \
                    .format("cloudFiles")\
                    .option("cloudFiles.format", "json")\
                    .option("pathGlobFilter", f"*{filename}*.json")\
                    .option("cloudFiles.inferColumnTypes", "false")\
                    .option("cloudFiles.schemaLocation", f"abfss://raw@bitcoindatalake.dfs.core.windows.net/schemas/schema_{filename}/")\
                    .load("abfss://raw@bitcoindatalake.dfs.core.windows.net/")

In [0]:
df_ohlc_data_last30days = get_ohlc_data("raw_ohlc_last30days")
df_ohlc_data_lastday = get_ohlc_data("raw_ohlc_lastday")

In [0]:
df_chartdata_last90days = get_chart_data("raw_chart_last90days")
df_chart_data_lastday = get_chart_data("raw_chart_lastday")

In [0]:
def load_df(df, table_name : str):
    df.writeStream \
    .format("delta") \
    .trigger(availableNow=True)\
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://bronze@bitcoindatalake.dfs.core.windows.net/checkpoints/{table_name}/")\
    .option("path", f"abfss://bronze@bitcoindatalake.dfs.core.windows.net/{table_name}/")\
    .toTable(table_name)

In [0]:
if mode == "initial_load":
    load_df(df_chartdata_last90days, "bronze_chart_last90days")
    load_df(df_ohlc_data_last30days, "bronze_ohlc_last30days")
else:
    load_df(df_chart_data_lastday, "bronze_chart_lastday")
    load_df(df_ohlc_data_lastday, "bronze_ohlc_lastday")
    load_df(df_chartdata_last90days, "bronze_chart_last90days")
    load_df(df_ohlc_data_last30days, "bronze_ohlc_last30days")


### load coin data

In [0]:
df_currency = get_currency("raw_currency")
df_coin = get_coin("raw_coin")

In [0]:
if mode == "initial_load":
    load_df(df_currency, "bronze_currency")
    load_df(df_coin, "bronze_coin")